# MedGemma 4B — Dermatoscopie Edge : Milestones & Architecture

**Objectif :** Construire un triptyque opérationnel combinant un modèle **MedGemma 4B** fine-tuné en dermatoscopie, un **harness méthodologique & clinique**, et un **wrapper applicatif Edge (100 % local)**, en intégrant les contraintes matérielles (Colab Google AI Pro) et réglementaires européennes (MDR / EU AI Act).

> **Date de cadrage :** 23 novembre 2026

---
## Milestone 1 — Infrastructure & Budget Google Colab (Google AI Pro)

### Ressources & Quotas
- **Quota mensuel :** ~200 Compute Units (CU) / mois inclus avec Google AI Pro.
- **GPU disponibles :**
  - **L4 (24 Go VRAM)** : ~1,5 CU/h (~130 h/mois) — idéal pour prototypage, texte pur (Unsloth) et inférence.
  - **A100 (40 Go VRAM)** : ~5,3 CU/h (~37 h/mois) — recommandé pour le fine-tuning **multimodal (images 896×896 + texte)**.
  - **A100 (80 Go VRAM)** : ~7,0 CU/h (~28 h/mois) — gros `batch_size` ou contextes longs.
  - **T4 (16 Go, gratuit)** : **à éviter** pour MedGemma (pas de `bfloat16` natif, instabilités `fp16` / NaN sur Gemma 3).
- **TPU disponibles :** `v5e-1` (16 Go, ~3 CU/h) et `v6e-1 Trillium` (32 Go HBM, ~4 CU/h). Pas d'accès aux pods multi-puces ni aux v5p/Ironwood sur Colab.

### Stack retenue
- **Choix principal : Hugging Face QLoRA (`transformers`, `peft`, `trl`, `bitsandbytes` / `unsloth`) sur GPU (A100 / L4).**
  - Support multimodal clé en main (notebook officiel Google Health).
  - Export direct vers `.safetensors`, GGUF (`llama.cpp` / Ollama / LM Studio), vLLM ou LiteRT.
- *Alternative TPU v6e-1 + KerasHub (JAX)* : pertinente uniquement pour du texte pur en LoRA `bfloat16` natif ou un déploiement cible sur Cloud TPU.

### Hygiène opérationnelle (pas d'exécution en arrière-plan sur AI Pro)
- [ ] Monter Google Drive systématiquement au démarrage.
- [ ] Configurer des checkpoints fréquents (`save_steps` tous les 50–100 steps) vers Drive.
- [ ] Déconnecter et supprimer le runtime dès la fin du run pour préserver les CU.

In [ ]:
# Vérification rapide du runtime GPU et montage Google Drive (à exécuter sur Colab)
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f'GPU détecté : {gpu_name} ({vram_gb:.1f} Go VRAM) | bfloat16 natif : {bf16_ok}')
else:
    print('Aucun GPU CUDA détecté. Sélectionner L4 ou A100 dans Exécution > Modifier le type d\'exécution.')

# Décommenter sur Colab pour monter Google Drive :
# from google.colab import drive
# drive.mount('/content/drive')
# CHECKPOINT_DIR = '/content/drive/MyDrive/MedGemma_Dermato/checkpoints'

---
## Milestone 2 — Données & Fine-Tuning Multimodal (MedGemma 4B)

### Spécifications MedGemma 4B (`google/medgemma-4b-it`)
- **Architecture multimodale native :** Encodeur de vision médical **`MedSigLIP`** (images normalisées en **896×896 px**, encodées en **256 tokens/image**) + LLM Gemma 3 4B.
- **Entrées / Sorties :** Texte + images 2D en entrée -> texte structuré (JSON, compte-rendu) en sortie. *(Pas d'audio natif ni de génération d'image).*

### Préparation du Dataset
- [ ] Sélectionner et télécharger les bases publiques de référence (**ISIC / HAM10000 / SLICE-3D**) — prévoir **2 à 15 Go** sur Drive.
- [ ] Garantir l'anonymisation stricte (zéro donnée patient réelle non anonymisée sur Colab grand public).
- [ ] Construire un split strict `train / val / test` (sans fuite par patient/lésion).
- [ ] Formater en **JSONL multimodal** orienté **description sémiologique structurée** (critères ABCD, *7-point checklist* ou *Chaos and Clues*) plutôt qu'une simple étiquette binaire.

### Entraînement
- [ ] Lancer le fine-tuning QLoRA 4-bit en `bfloat16` sur **A100 40 Go** (ou L4 avec `gradient_checkpointing` et `batch_size=1-2`).
- [ ] Sauvegarder les adaptateurs LoRA versionnés sur Drive (~100 à 250 Mo par expérience).

---
## Milestone 3 — Harness Méthodologique & Évaluation Clinique (Eval Harness)

Le harness est le cœur de la fiabilité du système — plus déterminant que le fine-tuning seul.

### 1. Garde-fou pré-analytique (Pre-flight check)
- [ ] Implémenter un contrôle automatique de qualité d'image avant inférence (flou, surexposition, bulles de gel, poils excessifs, cadrage).
- [ ] Refuser l'analyse et demander une reprise si l'image ne passe pas les seuils qualité.

### 2. Protocole de raisonnement forcé
- [ ] Imposer une sortie JSON stricte détaillant chaque critère sémiologique (asymétrie, réseau pigmentaire, globules, patron vasculaire, voile bleu-blanc) avant toute synthèse.

### 3. Banc d'évaluation verrouillé (Hold-out Test Set)
- [ ] Mesurer la **sensibilité sur les lésions malignes** (priorité absolue : minimiser les faux négatifs sur mélanome).
- [ ] Évaluer la **calibration et l'abstention** (capacité à signaler une incertitude élevée et renvoyer vers un avis spécialisé).
- [ ] Comparer systématiquement les métriques **avant et après quantification** (`BF16` vs `Q8_0` vs `Q4_K_M`).

---
## Milestone 4 — Export Edge & Quantification Optimale

| Précision | Poids sur disque | VRAM / RAM requise | Cible recommandée |
| :--- | :--- | :--- | :--- |
| **BF16 (Natif)** | ~8,5 Go | 10 – 12 Go | Référence clinique (PC cabinet GPU 12 Go ou serveur) |
| **Q8_0 + Vision FP16** | ~5,0 Go | 6,5 – 8 Go | **Meilleur compromis** (GPU 8 Go ou laptop 16 Go RAM) |
| **Q4_K_M + Vision FP16** | ~3,3 – 3,5 Go | 5 – 6 Go | Edge léger / Mobile (smartphone 12 Go RAM ou GPU 6 Go) |

### Règle d'or en dermatoscopie
> **Ne jamais quantifier l'encodeur visuel (`mmproj`) en 4-bit.** Conserver systématiquement `mmproj` en **FP16/BF16 (~850 Mo)** lors de la conversion GGUF pour ne pas dégrader la détection des micro-structures.

### Livrables
- [ ] Fusionner les adaptateurs LoRA (`merge_and_unload`).
- [ ] Exporter en GGUF **Q8_0** (avec `mmproj` FP16) pour déploiement local (`llama.cpp` / Ollama / LM Studio).
- [ ] Rejouer l'Eval Harness sur le modèle quantifié pour valider l'absence de régression clinique.

---
## Milestone 5 — Wrapper App Terrain (Edge Local)

### Principes Produit & UX
- **100 % Local / Hors-ligne :** Zéro donnée patient envoyée dans le cloud (levier RGPD / HDS majeur + coût d'inférence marginal nul).
- **Pas de chatbot :** Workflow 1-clic adapté au temps médical.

### Fonctionnalités clés
- [ ] Import direct depuis dermatoscope USB ou dossier local synchronisé.
- [ ] Exécution du pre-flight check qualité + inférence locale (FastAPI + `llama.cpp`/Ollama + UI locale).
- [ ] Affichage côte à côte : image dermatoscopique + grille sémiologique cochée + bloc texte prêt à copier-coller dans le DPI.
- [ ] **Human-in-the-loop :** Correction des critères par le praticien + saisie ultérieure du résultat anatomopathologique pour enrichir le dataset.

---
## Milestone 6 — Stratégie Réglementaire UE (MDR & EU AI Act)

### Contrainte structurelle
- Un logiciel d'aide au diagnostic dermatoscopique (suspicion de mélanome) relève du **MDR Classe IIa minimum** + **EU AI Act (Haut Risque)** : ~150k à 400k € et 18 à 36 mois en solo.
- **Piège à éviter :** Un simple disclaimer (*« Ceci n'est pas un dispositif médical »*) n'a aucune valeur juridique si l'app affiche un score de risque ou une classification clinique.
- **Rôles :** L'Organisme Notifié (BSI, TÜV, GMED) audite mais ne conseille pas ; l'accompagnement passe par un cabinet **QA/RA**.

### Décision stratégique à verrouiller dès le Jour 1
- [ ] **Option A — Hors-MDR par design :** Positionner le produit comme **simulateur / outil de formation continue** en dermatoscopie ou **contrôle qualité pré-analytique d'images** (télédermatologie).
- [ ] **Option B — Research Use Only (RUO) & B2B :** Utiliser le triptyque comme vitrine R&D / actif sous licence auprès d'une MedTech (déjà ISO 13485) ou d'un CHU/université qui porte le marquage CE.
- [ ] **Option C — Voie MedTech subventionnée :** Passer par un pôle/incubateur santé (ex. **BioWin**, incubateurs universitaires, SPW Recherche) et des subsides non-dilutifs pour financer un cabinet QA/RA.

In [ ]:
# Tableau de bord léger de suivi des milestones
milestones = [
    {'id': 'M1', 'titre': 'Setup Colab (AI Pro) + Drive + Checkpoints', 'statut': 'TODO'},
    {'id': 'M2', 'titre': 'Dataset ISIC JSONL + Fine-Tuning QLoRA MedGemma 4B', 'statut': 'TODO'},
    {'id': 'M3', 'titre': 'Eval Harness (Pre-flight + JSON sémiologique + Sensibilité)', 'statut': 'TODO'},
    {'id': 'M4', 'titre': 'Export GGUF Q8_0 (mmproj FP16) + Ré-évaluation post-quant', 'statut': 'TODO'},
    {'id': 'M5', 'titre': 'Wrapper App Edge Local (FastAPI + UI 1-clic + HITL)', 'statut': 'TODO'},
    {'id': 'M6', 'titre': 'Verrouillage positionnement UE (Hors-MDR / RUO B2B / Subsides)', 'statut': 'TODO'},
]

for m in milestones:
    print(f"[{m['statut']:^11}] {m['id']} — {m['titre']}")